<a href="https://colab.research.google.com/github/Architag1503/Colab/blob/main/AprioriAlgorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Scratch Version**

In [1]:
from itertools import combinations

# -------------------------------
# Step 1: Dataset
# -------------------------------
transactions = [
    ['A', 'B', 'C', 'D'],
    ['A', 'C', 'D'],
    ['B', 'C', 'E'],
    ['A', 'B', 'C', 'E'],
    ['B', 'E'],
    ['A', 'B', 'C', 'E'],
    ['A', 'B', 'D'],
    ['B', 'C', 'D', 'E']
]

In [2]:
# -------------------------------
# Step 2: Support Function
# -------------------------------
def get_support(transactions, itemset):
    count = 0
    for t in transactions:
        if set(itemset).issubset(set(t)):
            count += 1
    return count

In [3]:
# -------------------------------
# Step 3: Generate Candidates
# -------------------------------
def generate_candidates(prev_L, k):
    candidates = set()
    prev_L = list(prev_L)

    for i in range(len(prev_L)):
        for j in range(i + 1, len(prev_L)):
            union = tuple(sorted(set(prev_L[i]) | set(prev_L[j])))
            if len(union) == k:
                candidates.add(union)

    return candidates

In [4]:
# -------------------------------
# Step 4: Apriori Algorithm
# -------------------------------
def apriori(transactions, min_support):

    # Get unique items
    items = set()
    for t in transactions:
        for item in t:
            items.add((item,))

    # L1
    L1 = set()
    support_data = {}

    for item in items:
        sup = get_support(transactions, item)
        if sup >= min_support:
            L1.add(item)
            support_data[item] = sup

    L = [L1]
    k = 2

    # Generate Lk
    while True:
        Ck = generate_candidates(L[k-2], k)
        Lk = set()

        for candidate in Ck:
            sup = get_support(transactions, candidate)
            if sup >= min_support:
                Lk.add(candidate)
                support_data[candidate] = sup

        if not Lk:
            break

        L.append(Lk)
        k += 1

    return L, support_data

In [5]:
# -------------------------------
# Step 5: Generate Association Rules
# -------------------------------
def generate_rules(L, support_data, min_confidence):
    rules = []

    for i in range(1, len(L)):
        for itemset in L[i]:
            for r in range(1, len(itemset)):
                subsets = combinations(itemset, r)

                for antecedent in subsets:
                    antecedent = tuple(sorted(antecedent))
                    consequent = tuple(sorted(set(itemset) - set(antecedent)))

                    if antecedent in support_data:
                        confidence = support_data[itemset] / support_data[antecedent]

                        if confidence >= min_confidence:
                            rules.append((antecedent, consequent, confidence))

    return rules

In [6]:
# -------------------------------
# Step 6: Run Algorithm
# -------------------------------
min_support = 3
min_confidence = 0.6

L, support_data = apriori(transactions, min_support)

In [7]:
# -------------------------------
# Step 7: Print Frequent Itemsets
# -------------------------------
print("🔷 Frequent Itemsets:\n")
for i, level in enumerate(L):
    print(f"L{i+1}:")
    for item in level:
        print(f"{item} -> Support: {support_data[item]}")
    print()

🔷 Frequent Itemsets:

L1:
('E',) -> Support: 5
('C',) -> Support: 6
('D',) -> Support: 4
('A',) -> Support: 5
('B',) -> Support: 7

L2:
('B', 'C') -> Support: 5
('B', 'E') -> Support: 5
('A', 'C') -> Support: 4
('B', 'D') -> Support: 3
('C', 'E') -> Support: 4
('C', 'D') -> Support: 3
('A', 'D') -> Support: 3
('A', 'B') -> Support: 4

L3:
('A', 'B', 'C') -> Support: 3
('B', 'C', 'E') -> Support: 4



In [8]:
# -------------------------------
# Step 8: Generate & Print Rules
# -------------------------------
rules = generate_rules(L, support_data, min_confidence)

print("🔷 Association Rules:\n")
for antecedent, consequent, confidence in rules:
    print(f"{antecedent} -> {consequent}  (Confidence: {confidence:.2f})")

🔷 Association Rules:

('B',) -> ('C',)  (Confidence: 0.71)
('C',) -> ('B',)  (Confidence: 0.83)
('B',) -> ('E',)  (Confidence: 0.71)
('E',) -> ('B',)  (Confidence: 1.00)
('A',) -> ('C',)  (Confidence: 0.80)
('C',) -> ('A',)  (Confidence: 0.67)
('D',) -> ('B',)  (Confidence: 0.75)
('C',) -> ('E',)  (Confidence: 0.67)
('E',) -> ('C',)  (Confidence: 0.80)
('D',) -> ('C',)  (Confidence: 0.75)
('A',) -> ('D',)  (Confidence: 0.60)
('D',) -> ('A',)  (Confidence: 0.75)
('A',) -> ('B',)  (Confidence: 0.80)
('A',) -> ('B', 'C')  (Confidence: 0.60)
('A', 'B') -> ('C',)  (Confidence: 0.75)
('A', 'C') -> ('B',)  (Confidence: 0.75)
('B', 'C') -> ('A',)  (Confidence: 0.60)
('C',) -> ('B', 'E')  (Confidence: 0.67)
('E',) -> ('B', 'C')  (Confidence: 0.80)
('B', 'C') -> ('E',)  (Confidence: 0.80)
('B', 'E') -> ('C',)  (Confidence: 0.80)
('C', 'E') -> ('B',)  (Confidence: 1.00)


# **Library Version**

In [9]:
!pip install mlxtend

In [10]:
# Step 2: Import libraries
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
# Step 3: Define dataset
transactions = [
    ['A', 'B', 'C', 'D'],
    ['A', 'C', 'D'],
    ['B', 'C', 'E'],
    ['A', 'B', 'C', 'E'],
    ['B', 'E'],
    ['A', 'B', 'C', 'E'],
    ['A', 'B', 'D'],
    ['B', 'C', 'D', 'E']
]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [13]:
# Step 4: Convert transactions → One-hot encoding
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_)

print("🔷 One-Hot Encoded Data:\n")
print(df)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🔷 One-Hot Encoded Data:

       A      B      C      D      E
0   True   True   True   True  False
1   True  False   True   True  False
2  False   True   True  False   True
3   True   True   True  False   True
4  False   True  False  False   True
5   True   True   True  False   True
6   True   True  False   True  False
7  False   True   True   True   True


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [14]:
# Step 5: Apply Apriori Algorithm
# min_support = 3/8 because minimum count = 3
frequent_itemsets = apriori(df, min_support=3/8, use_colnames=True)

print("\n🔷 Frequent Itemsets:\n")
print(frequent_itemsets)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


🔷 Frequent Itemsets:

    support   itemsets
0     0.625        (A)
1     0.875        (B)
2     0.750        (C)
3     0.500        (D)
4     0.625        (E)
5     0.500     (A, B)
6     0.500     (A, C)
7     0.375     (A, D)
8     0.625     (C, B)
9     0.375     (B, D)
10    0.625     (B, E)
11    0.375     (C, D)
12    0.500     (C, E)
13    0.375  (A, B, C)
14    0.500  (C, B, E)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [15]:
# Step 6: Generate Association Rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

print("\n🔷 Association Rules:\n")
print(rules)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


🔷 Association Rules:

   antecedents consequents  antecedent support  consequent support  support  \
0          (A)         (B)               0.625               0.875    0.500   
1          (A)         (C)               0.625               0.750    0.500   
2          (C)         (A)               0.750               0.625    0.500   
3          (A)         (D)               0.625               0.500    0.375   
4          (D)         (A)               0.500               0.625    0.375   
5          (C)         (B)               0.750               0.875    0.625   
6          (B)         (C)               0.875               0.750    0.625   
7          (D)         (B)               0.500               0.875    0.375   
8          (B)         (E)               0.875               0.625    0.625   
9          (E)         (B)               0.625               0.875    0.625   
10         (D)         (C)               0.500               0.750    0.375   
11         (C)         (E)   

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [16]:
# Step 7: Display only useful columns (clean output)
print("\n🔷 Cleaned Rules Output:\n")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


🔷 Cleaned Rules Output:

   antecedents consequents  support  confidence      lift
0          (A)         (B)    0.500    0.800000  0.914286
1          (A)         (C)    0.500    0.800000  1.066667
2          (C)         (A)    0.500    0.666667  1.066667
3          (A)         (D)    0.375    0.600000  1.200000
4          (D)         (A)    0.375    0.750000  1.200000
5          (C)         (B)    0.625    0.833333  0.952381
6          (B)         (C)    0.625    0.714286  0.952381
7          (D)         (B)    0.375    0.750000  0.857143
8          (B)         (E)    0.625    0.714286  1.142857
9          (E)         (B)    0.625    1.000000  1.142857
10         (D)         (C)    0.375    0.750000  1.000000
11         (C)         (E)    0.500    0.666667  1.066667
12         (E)         (C)    0.500    0.800000  1.066667
13      (A, B)         (C)    0.375    0.750000  1.000000
14      (A, C)         (B)    0.375    0.750000  0.857143
15      (C, B)         (A)    0.375    0.60000

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag